In [8]:
import pandas as pd
import re
import pandas as pd
from tqdm.auto import tqdm
from dotenv import load_dotenv
import os
from pinecone import Pinecone
from langchain_ollama import OllamaEmbeddings
import pickle
from langchain_community.retrievers import PineconeHybridSearchRetriever
import numpy as np

load_dotenv()

True

In [7]:
pine_api_key = os.getenv("PINECONE_API_KEY")

In [38]:
sys_prompt = """
You are a retrieval-grounded Quran QA assistant (English). You MUST answer using ONLY the provided retrieved context. Treat the retrieved context as the ONLY source of truth. The retrieved context may include Qur'an translation text and Tafsir (e.g., Tafsir al-Jalalayn).

Rules (strict):
1) Use ONLY the retrieved context. Do NOT use outside knowledge, memory, or guesses.
2) Do NOT invent or guess: verse numbers, surah names, citations, quotes, narrations, or any details not explicitly present in the retrieved context.
3) If you cannot answer the question with explicit support from the retrieved context, you MUST output exactly:
   Answer: INSUFFICIENT_CONTEXT
   Evidence: NONE
   Queries: <3–7 suggested retrieval keywords/queries>
   Then stop.
4) Every factual claim in your Answer MUST be supported by Evidence.
   - Evidence MUST be copied verbatim from the retrieved context (exact substring).
   - Do NOT paraphrase evidence.
   - If you cannot provide a verbatim quote that supports the claim, output INSUFFICIENT_CONTEXT (do not infer).
5) Distinguish sources clearly:
   - If the supporting quote is Qur'an translation text, label it as [Quran].
   - If the supporting quote is tafsir, label it as [Tafsir] and treat it as explanation, not a direct Qur'an quote.
6) Citations MUST be taken from the retrieved context metadata. Only cite a surah/ayah range if it is explicitly present in the retrieved context. If the context does not include surah/ayah metadata, do NOT guess it; instead output INSUFFICIENT_CONTEXT.
7) If retrieved passages are ambiguous or contradictory, the Answer MUST start with:
   AMBIGUOUS:
   and you must provide evidence quotes for each interpretation/view.
8) Keep the tone respectful and neutral. Do not issue fatwas or absolute legal rulings. If the question requires scholarly/legal judgment beyond the retrieved context, output INSUFFICIENT_CONTEXT.

Output format (STRICT: no extra lines, no extra commentary):
Answer: <ONE sentence max, unless INSUFFICIENT_CONTEXT>
Evidence:
- [Quran|Tafsir] "(Surah X Ayah A-B)" "<verbatim quote from retrieved context>"
- [Quran|Tafsir] "(Surah X Ayah A-B)" "<verbatim quote from retrieved context>"
- ...
"""

In [12]:
df = pd.read_csv("question_answering.csv")
df

,Unnamed: 0,question_en,answer_en,chapter_no,verse_list
0,0,What is the only book that is free from any do...,This is the Book of Allah . The evidence: 'Thi...,2,[1 2]
1,1,Are the fruits of paradise similar to the frui...,"Yes, and the evidence: 'And give good news to ...",2,[25]
2,2,How many deaths and how many lives do humans h...,"And you were dead, and He gave you life, then ...",2,[28]
3,3,How many heavens are there?,He it is Who created for you all that is in th...,2,[29]
4,4,"What did Adam learn from Allah , which was no...","He taught Adam the names of all things, then H...",2,[31]
...,...,...,...,...,...
1186,1218,Who is the wife of Ibrahim who laughed when s...,She is Sarah.,11,[71]
1187,1219,"Indeed, Abraham was forbearing, often turning ...","Abraham, peace be upon him, is patient, dislik...",11,[75]
1188,1220,Why did Lot grieve when the guests arrived at...,He feared for them because they were handsome-...,11,[77]
1189,1221,"What does Lut mean by his statement, 'If only...","If I had power and supporters among you, or if...",11,[80]


In [13]:
embeddings = OllamaEmbeddings(
    model="qwen3-embedding:8b",
    base_url=os.environ.get("OLLAMA_URL", "http://localhost:11434"),
)

In [39]:
def parse_verse_list_str(s: str):
    return [int(x) for x in re.findall(r"\d+", str(s))]

def chunk_hits_any_ayah(chunk_meta: dict, gt_surah: int, gt_ayah_list) -> bool:
    try:
        surah = int(chunk_meta.get("surah_no"))
        a0 = int(chunk_meta.get("ayah_start"))
        a1 = int(chunk_meta.get("ayah_end"))
    except Exception:
        return False

    if surah != int(gt_surah):
        return False

    for ay in gt_ayah_list:
        if a0 <= int(ay) <= a1:
            return True
    return False

def _safe_meta(d):
    if d is None:
        return None
    m = getattr(d, "metadata", None)
    return dict(m) if isinstance(m, dict) else m

def eval_hit_at_k(
    qa_df: pd.DataFrame,
    retriever,
    k: int = 15,
    alpha: float = 0.7,
    question_col: str = "question",
    gt_surah_col: str = "surah_no",
    gt_verses_col: str = "verses",
):
    if hasattr(retriever, "top_k"):
        retriever.top_k = k
    if hasattr(retriever, "alpha"):
        retriever.alpha = alpha

    rows = []
    hits = 0

    for idx, row in tqdm(qa_df.iterrows(), total=len(qa_df), desc=f"Evaluating Hit@{k}"):
        q = str(row[question_col])
        gt_surah = int(row[gt_surah_col])
        gt_verses = parse_verse_list_str(row[gt_verses_col])

        docs = retriever.invoke(q) or []
        docs = list(docs)[:k]

        # store ALL retrievals' metadata (always)
        retrieved_docs_meta = [_safe_meta(d) for d in docs]

        hit = False
        hit_rank = None
        hit_doc = None

        for rank, d in enumerate(docs, start=1):
            if chunk_hits_any_ayah(getattr(d, "metadata", {}) or {}, gt_surah, gt_verses):
                hit = True
                hit_rank = rank
                hit_doc = d
                break

        hits += int(hit)

        rows.append({
            "row_idx": idx,
            "question": q,
            "gt_surah": gt_surah,
            "gt_verses": gt_verses,
            "hit": hit,
            "hit_rank": hit_rank,
            "hit_doc_meta": _safe_meta(hit_doc),           # None if no hit
            "retrieved_docs_meta": retrieved_docs_meta,    # list of k metas (or fewer)
        })

    report = pd.DataFrame(rows)
    hit_at_k = hits / len(qa_df) if len(qa_df) else 0.0
    return hit_at_k, report

In [18]:
pc = Pinecone(api_key=pine_api_key)
index_jal = pc.Index("quran-tafseer-jal")

In [17]:
with open("bm25_quran_jal.pkl", "rb") as f:
    bm25_jal = pickle.load(f)

In [25]:
NAMESPACE = "quran_jal"

retriever_jal = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_jal,
    index=index_jal,
    namespace=NAMESPACE,
    top_k=10,
    alpha=0.7,
)

docs = retriever_jal.invoke("What does the Quran say about patience?")
print(len(docs))
print(docs[0].metadata)
print(docs[0].page_content[:400])
print("_________________________________________________________________")
print(docs[1].metadata)
print(docs[1].page_content[:400])


10
{'ayah_end': 153.0, 'ayah_start': 153.0, 'surah_no': 2.0, 'score': 0.632772565}
ayah: o believers! seek comfort in patience and prayer. allah is truly with those who are patient.
tafseer: o you who believe seek help regarding the hereafter through patience in obedience and afflictions and prayer he singles it out for mention on account of its frequency and its greatness; surely god is with the patient helping them.
_________________________________________________________________
{'ayah_end': 5.0, 'ayah_start': 5.0, 'surah_no': 70.0, 'score': 0.617648721}
ayah: so endure ˹this denial, o prophet,˺ with beautiful patience.
tafseer: so be patient - this was revealed before he the prophet was commanded to fight - with a graceful patience that is one in which there is no anguish.


In [40]:
hit15_jal, report = eval_hit_at_k(
    qa_df=df,
    retriever=retriever_jal,
    k=15,
    alpha=0.7,
    question_col="question_en",
    gt_surah_col="chapter_no",
    gt_verses_col="verse_list",
)

print("Hit@15:", hit15_jal)
print(report["hit"].value_counts())
report.head()


Evaluating Hit@15:   0%|          | 0/1191 [00:00<?, ?it/s]

Hit@15: 0.7439126784214946
hit
True     886
False    305
Name: count, dtype: int64


,row_idx,question,gt_surah,gt_verses,hit,hit_rank,hit_doc_meta,retrieved_docs_meta
0,0,What is the only book that is free from any do...,2,"[1, 2]",True,1.0,"{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_no...","[{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_n..."
1,1,Are the fruits of paradise similar to the frui...,2,[25],True,5.0,"{'ayah_end': 25.0, 'ayah_start': 25.0, 'surah_...","[{'ayah_end': 42.0, 'ayah_start': 42.0, 'surah..."
2,2,How many deaths and how many lives do humans h...,2,[28],False,NaN,None,"[{'ayah_end': 34.0, 'ayah_start': 34.0, 'surah..."
3,3,How many heavens are there?,2,[29],True,6.0,"{'ayah_end': 29.0, 'ayah_start': 29.0, 'surah_...","[{'ayah_end': 105.0, 'ayah_start': 105.0, 'sur..."
4,4,"What did Adam learn from Allah , which was no...",2,[31],True,2.0,"{'ayah_end': 31.0, 'ayah_start': 31.0, 'surah_...","[{'ayah_end': 33.0, 'ayah_start': 33.0, 'surah..."


In [41]:
report

,row_idx,question,gt_surah,gt_verses,hit,hit_rank,hit_doc_meta,retrieved_docs_meta
0,0,What is the only book that is free from any do...,2,"[1, 2]",True,1.0,"{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_no...","[{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_n..."
1,1,Are the fruits of paradise similar to the frui...,2,[25],True,5.0,"{'ayah_end': 25.0, 'ayah_start': 25.0, 'surah_...","[{'ayah_end': 42.0, 'ayah_start': 42.0, 'surah..."
2,2,How many deaths and how many lives do humans h...,2,[28],False,NaN,None,"[{'ayah_end': 34.0, 'ayah_start': 34.0, 'surah..."
3,3,How many heavens are there?,2,[29],True,6.0,"{'ayah_end': 29.0, 'ayah_start': 29.0, 'surah_...","[{'ayah_end': 105.0, 'ayah_start': 105.0, 'sur..."
4,4,"What did Adam learn from Allah , which was no...",2,[31],True,2.0,"{'ayah_end': 31.0, 'ayah_start': 31.0, 'surah_...","[{'ayah_end': 33.0, 'ayah_start': 33.0, 'surah..."
...,...,...,...,...,...,...,...,...
1186,1186,Who is the wife of Ibrahim who laughed when s...,11,[71],True,1.0,"{'ayah_end': 71.0, 'ayah_start': 71.0, 'surah_...","[{'ayah_end': 71.0, 'ayah_start': 71.0, 'surah..."
1187,1187,"Indeed, Abraham was forbearing, often turning ...",11,[75],True,1.0,"{'ayah_end': 75.0, 'ayah_start': 75.0, 'surah_...","[{'ayah_end': 75.0, 'ayah_start': 75.0, 'surah..."
1188,1188,Why did Lot grieve when the guests arrived at...,11,[77],True,1.0,"{'ayah_end': 77.0, 'ayah_start': 77.0, 'surah_...","[{'ayah_end': 77.0, 'ayah_start': 77.0, 'surah..."
1189,1189,"What does Lut mean by his statement, 'If only...",11,[80],True,1.0,"{'ayah_end': 80.0, 'ayah_start': 80.0, 'surah_...","[{'ayah_end': 80.0, 'ayah_start': 80.0, 'surah..."


In [42]:
report["answer_en"] = df["answer_en"]
report

,row_idx,question,gt_surah,gt_verses,hit,hit_rank,hit_doc_meta,retrieved_docs_meta,answer_en
0,0,What is the only book that is free from any do...,2,"[1, 2]",True,1.0,"{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_no...","[{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_n...",This is the Book of Allah . The evidence: 'Thi...
1,1,Are the fruits of paradise similar to the frui...,2,[25],True,5.0,"{'ayah_end': 25.0, 'ayah_start': 25.0, 'surah_...","[{'ayah_end': 42.0, 'ayah_start': 42.0, 'surah...","Yes, and the evidence: 'And give good news to ..."
2,2,How many deaths and how many lives do humans h...,2,[28],False,NaN,None,"[{'ayah_end': 34.0, 'ayah_start': 34.0, 'surah...","And you were dead, and He gave you life, then ..."
3,3,How many heavens are there?,2,[29],True,6.0,"{'ayah_end': 29.0, 'ayah_start': 29.0, 'surah_...","[{'ayah_end': 105.0, 'ayah_start': 105.0, 'sur...",He it is Who created for you all that is in th...
4,4,"What did Adam learn from Allah , which was no...",2,[31],True,2.0,"{'ayah_end': 31.0, 'ayah_start': 31.0, 'surah_...","[{'ayah_end': 33.0, 'ayah_start': 33.0, 'surah...","He taught Adam the names of all things, then H..."
...,...,...,...,...,...,...,...,...,...
1186,1186,Who is the wife of Ibrahim who laughed when s...,11,[71],True,1.0,"{'ayah_end': 71.0, 'ayah_start': 71.0, 'surah_...","[{'ayah_end': 71.0, 'ayah_start': 71.0, 'surah...",She is Sarah.
1187,1187,"Indeed, Abraham was forbearing, often turning ...",11,[75],True,1.0,"{'ayah_end': 75.0, 'ayah_start': 75.0, 'surah_...","[{'ayah_end': 75.0, 'ayah_start': 75.0, 'surah...","Abraham, peace be upon him, is patient, dislik..."
1188,1188,Why did Lot grieve when the guests arrived at...,11,[77],True,1.0,"{'ayah_end': 77.0, 'ayah_start': 77.0, 'surah_...","[{'ayah_end': 77.0, 'ayah_start': 77.0, 'surah...",He feared for them because they were handsome-...
1189,1189,"What does Lut mean by his statement, 'If only...",11,[80],True,1.0,"{'ayah_end': 80.0, 'ayah_start': 80.0, 'surah_...","[{'ayah_end': 80.0, 'ayah_start': 80.0, 'surah...","If I had power and supporters among you, or if..."


In [43]:
report.loc[report["hit"] == False, "answer_en"] = "Not Found"
report

,row_idx,question,gt_surah,gt_verses,hit,hit_rank,hit_doc_meta,retrieved_docs_meta,answer_en
0,0,What is the only book that is free from any do...,2,"[1, 2]",True,1.0,"{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_no...","[{'ayah_end': 2.0, 'ayah_start': 2.0, 'surah_n...",This is the Book of Allah . The evidence: 'Thi...
1,1,Are the fruits of paradise similar to the frui...,2,[25],True,5.0,"{'ayah_end': 25.0, 'ayah_start': 25.0, 'surah_...","[{'ayah_end': 42.0, 'ayah_start': 42.0, 'surah...","Yes, and the evidence: 'And give good news to ..."
2,2,How many deaths and how many lives do humans h...,2,[28],False,NaN,None,"[{'ayah_end': 34.0, 'ayah_start': 34.0, 'surah...",Not Found
3,3,How many heavens are there?,2,[29],True,6.0,"{'ayah_end': 29.0, 'ayah_start': 29.0, 'surah_...","[{'ayah_end': 105.0, 'ayah_start': 105.0, 'sur...",He it is Who created for you all that is in th...
4,4,"What did Adam learn from Allah , which was no...",2,[31],True,2.0,"{'ayah_end': 31.0, 'ayah_start': 31.0, 'surah_...","[{'ayah_end': 33.0, 'ayah_start': 33.0, 'surah...","He taught Adam the names of all things, then H..."
...,...,...,...,...,...,...,...,...,...
1186,1186,Who is the wife of Ibrahim who laughed when s...,11,[71],True,1.0,"{'ayah_end': 71.0, 'ayah_start': 71.0, 'surah_...","[{'ayah_end': 71.0, 'ayah_start': 71.0, 'surah...",She is Sarah.
1187,1187,"Indeed, Abraham was forbearing, often turning ...",11,[75],True,1.0,"{'ayah_end': 75.0, 'ayah_start': 75.0, 'surah_...","[{'ayah_end': 75.0, 'ayah_start': 75.0, 'surah...","Abraham, peace be upon him, is patient, dislik..."
1188,1188,Why did Lot grieve when the guests arrived at...,11,[77],True,1.0,"{'ayah_end': 77.0, 'ayah_start': 77.0, 'surah_...","[{'ayah_end': 77.0, 'ayah_start': 77.0, 'surah...",He feared for them because they were handsome-...
1189,1189,"What does Lut mean by his statement, 'If only...",11,[80],True,1.0,"{'ayah_end': 80.0, 'ayah_start': 80.0, 'surah_...","[{'ayah_end': 80.0, 'ayah_start': 80.0, 'surah...","If I had power and supporters among you, or if..."


In [45]:
report.to_csv("qa_with_context.csv", index=False, encoding="utf-8-sig")